In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression,SGDRegressor

In [ ]:
df = pd.read_csv('bangladesh_student_performance_updated.csv')

In [ ]:
df.shape

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
df['M_Edu'].unique()
df['time_friends'].unique()
df['age'].unique()
df['tuition_fee'].unique()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.hist(figsize=(10,8), bins= 20)

plt.suptitle("Distribution of numerical columns")

# Matrix Co-relation

In [ ]:
num_col = df.select_dtypes(include=["number"]).columns
print(num_col)

df[num_col].corr()

# Co-relattion heatmap

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    df[num_col].corr(),
    annot=True,
    cmap ="coolwarm"
)
plt.title("Correlation heatmap for numerical columns")
plt.show()

In [ ]:
sns.boxplot(data=df , x ="ssc_result")

### pre processing


In [ ]:
nominal_cate =['gender','address','famsize','Pstatus','relationship','smoker','M_Job','F_Job']
#ordinal_cate = ['M_Edu','F_Edu','time_friends']
#M_edu_order =['0','1','2','3','4']                  # eikane ordinal category gula bad daoar reason hocce sob ordinal values gula int64 a
#F_edu_order =['0','1','2','3','4']
#time_friends_order =['5','4','3','2','1']


num_col = ['age','tuition_fee','ssc_result']

In [ ]:
numerical_transformers = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)
numerical_transformers

In [ ]:
nominal_transformers = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,handle_unknown="ignore"))
    ]
)
nominal_transformers

### Final model


In [ ]:
preprocessor = ColumnTransformer(
   transformers=[
       ('numerical',numerical_transformers,num_col),
       ('nominal',nominal_transformers,nominal_cate),
       #('ordinal',ordinal_transformers,ordinal_cate)
   ],
   remainder="passthrough"
)
preprocessor

In [ ]:
X = df.drop(['date','hsc_result'],axis=1)    # feature values
y = df['hsc_result']                 # target values
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2 , random_state=42)


In [ ]:
X_train

# Linear Regression

In [ ]:
lr_pipe = Pipeline (
    steps =[
        ('preprocessor',preprocessor),
        ('model',LinearRegression())
    ]
)
lr_pipe.fit(X_train,y_train)


In [ ]:
lr_pipe.predict(X_test)

# SGD pipeline

In [ ]:
sgd_pipe = Pipeline(
    steps =[
        ('preprocessor',preprocessor),
        ('model',SGDRegressor())
    ]
)
sgd_pipe.fit(X_train,y_train)

In [ ]:
sgd_pipe.predict(X_test)

# Model performance

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [ ]:
# linear regression

y_pred = lr_pipe.predict(X_test)
print(f"MSE :{round(mean_squared_error(y_test,y_pred),4)}")
print(f"R2 score :{round(r2_score(y_test,y_pred),4)}")
print(f"RMSE :{round(root_mean_squared_error(y_test,y_pred),4)}")
print(f"MAE :{round(mean_absolute_error(y_test,y_pred),4)}")

In [ ]:
# sgd regression
y_pred = sgd_pipe.predict(X_test)
print(f"MSE :{round(mean_squared_error(y_test,y_pred),4)}")
print(f"R2 score :{round(r2_score(y_test,y_pred),4)}")
print(f"RMSE :{round(root_mean_squared_error(y_test,y_pred),4)}")
print(f"MAE :{round(mean_absolute_error(y_test,y_pred),4)}")